In [1]:
import pandas as pd
import numpy as np
import os
import re
import math
import time
import random

In [2]:
df = pd.read_csv("../../Feature_Generation/calculated_features/ibs features.csv", index_col=0)
# df.drop("Unnamed: 0", axis=1, inplace=True)
df.head()

,entityNo,ownerID,numCurrentLiveBusinesses,numCurrentLiquidatedBusinesses,numCurrentDissolvedBusinesses,numDirectors,numAlternateDirectors,numShareholders,numSecretaries,numActiveCharges,...,totalamountOfBankruptcyRecord_Others_Within12Months_Individualplantiff,numberOfBankruptcyRecord_Others_Within12Months_Individualplantiff,totalamountOfBankruptcyRecord_Others_Within24Months_Companyplantiff,numberOfBankruptcyRecord_Others_Within24Months_Companyplantiff,totalamountOfBankruptcyRecord_Others_Within24Months_Individualplantiff,numberOfBankruptcyRecord_Others_Within24Months_Individualplantiff,totalamountOfBankruptcyRecord_Others_Within36Months_Companyplantiff,numberOfBankruptcyRecord_Others_Within36Months_Companyplantiff,totalamountOfBankruptcyRecord_Others_Within36Months_Individualplantiff,numberOfBankruptcyRecord_Others_Within36Months_Individualplantiff
0,T05LL7197F,J663XXXXQ,0,0,0,3,1,2,0,1,...,69694.0,1,22499.0,1,69694.0,1,22499.0,1,69694.0,1
1,200573534H,J663XXXXQ,0,0,0,3,2,1,1,2,...,69694.0,1,22499.0,1,69694.0,1,22499.0,1,69694.0,1
2,84380899Q,J663XXXXQ,0,0,0,2,3,0,0,3,...,69694.0,1,22499.0,1,69694.0,1,22499.0,1,69694.0,1
3,T12CL3953B,E778XXXXM,0,0,0,0,3,0,3,1,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0
4,T12CP1418L,E778XXXXM,0,0,0,0,3,2,3,1,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0


In [3]:
df["target"] = [random.randint(0,1) for i in range(len(df))]
df["target"]

0     1
1     1
2     0
3     0
4     1
     ..
62    1
63    1
64    0
65    1
66    1
Name: target, Length: 67, dtype: int64

In [4]:
def is_monotonic(temp_series):
    return all(temp_series[i] <= temp_series[i + 1] for i in range(len(temp_series) - 1)) or all(temp_series[i] >= temp_series[i + 1] for i in range(len(temp_series) - 1))

In [5]:
def prepare_bins(bin_data, c_i, target_col, max_bins):
    force_bin = True
    binned = False
    remarks = np.nan
    # ----------------- Monotonic binning -----------------
    for n_bins in range(max_bins, 2, -1):
        try:
            bin_data[c_i + "_bins"] = pd.qcut(bin_data[c_i], n_bins, duplicates="drop")
            monotonic_series = bin_data.groupby(c_i + "_bins")[target_col].mean().reset_index(drop=True)
            if is_monotonic(monotonic_series):
                force_bin = False
                binned = True
                remarks = "binned monotonically"
                break
        except:
            pass
    # ----------------- Force binning -----------------
    # creating 2 bins forcefully because 2 bins will always be monotonic
    if force_bin or (c_i + "_bins" in bin_data and bin_data[c_i + "_bins"].nunique() < 2):
        _min=bin_data[c_i].min()
        _mean=bin_data[c_i].mean()
        _max=bin_data[c_i].max()
        bin_data[c_i + "_bins"] = pd.cut(bin_data[c_i], [_min, _mean, _max], include_lowest=True)
        if bin_data[c_i + "_bins"].nunique() == 2:
            binned = True
            remarks = "binned forcefully"
    
    if binned:
        return c_i + "_bins", remarks, bin_data[[c_i, c_i+"_bins", target_col]].copy()
    else:
        remarks = "couldn't bin"
        return c_i, remarks, bin_data[[c_i, target_col]].copy()

In [6]:
def iv_woe_4iter(binned_data, target_col, class_col):
    if "_bins" in class_col:
        binned_data[class_col] = binned_data[class_col].cat.add_categories(['Missing'])
        binned_data[class_col] = binned_data[class_col].fillna("Missing")
        temp_groupby = binned_data.groupby(class_col).agg({class_col.replace("_bins", ""):["min", "max"],
                                                           target_col: ["count", "sum", "mean"]}).reset_index()
    else:
        binned_data[class_col] = binned_data[class_col].fillna("Missing")
        temp_groupby = binned_data.groupby(class_col).agg({class_col:["first", "first"],
                                                           target_col: ["count", "sum", "mean"]}).reset_index()
    
    temp_groupby.columns = ["sample_class", "min_value", "max_value", "sample_count", "event_count", "event_rate"]
    temp_groupby["non_event_count"] = temp_groupby["sample_count"] - temp_groupby["event_count"]
    temp_groupby["non_event_rate"] = 1 - temp_groupby["event_rate"]
    temp_groupby = temp_groupby[["sample_class", "min_value", "max_value", "sample_count",
                                 "non_event_count", "non_event_rate", "event_count", "event_rate"]]
    
    if "_bins" not in class_col and "Missing" in temp_groupby["min_value"]:
        temp_groupby["min_value"] = temp_groupby["min_value"].replace({"Missing": np.nan})
        temp_groupby["max_value"] = temp_groupby["max_value"].replace({"Missing": np.nan})
    temp_groupby["feature"] = class_col
    if "_bins" in class_col:
        temp_groupby["sample_class_label"]=temp_groupby["sample_class"].replace({"Missing": np.nan}).astype('category').cat.codes.replace({-1: np.nan})
    else:
        temp_groupby["sample_class_label"]=np.nan
    temp_groupby = temp_groupby[["feature", "sample_class", "sample_class_label", "sample_count", "min_value", "max_value",
                                 "non_event_count", "non_event_rate", "event_count", "event_rate"]]
    
    """
    **********get distribution of good and bad
    """
    temp_groupby['distbn_non_event'] = temp_groupby["non_event_count"]/temp_groupby["non_event_count"].sum()
    temp_groupby['distbn_event'] = temp_groupby["event_count"]/temp_groupby["event_count"].sum()

    temp_groupby['woe'] = np.log(temp_groupby['distbn_non_event'] / temp_groupby['distbn_event'])
    temp_groupby['iv'] = (temp_groupby['distbn_non_event'] - temp_groupby['distbn_event']) * temp_groupby['woe']
    
    temp_groupby["woe"] = temp_groupby["woe"].replace([np.inf,-np.inf],0)
    temp_groupby["iv"] = temp_groupby["iv"].replace([np.inf,-np.inf],0)
    
    return temp_groupby

In [7]:
def var_iter(data, target_col, max_bins):
    woe_iv = pd.DataFrame()
    remarks_list = []
    for c_i in data.columns:
        if c_i not in [target_col]:
            # check if binning is required. if yes, then prepare bins and calculate woe and iv.
            """
            ----logic---
            binning is done only when feature is continuous and non-binary.
            Note: Make sure dtype of continuous columns in dataframe is not object.
            """
            c_i_start_time=time.time()
            if np.issubdtype(data[c_i], np.number) and data[c_i].nunique() > 2:
                class_col, remarks, binned_data = prepare_bins(data[[c_i, target_col]].copy(), c_i, target_col, max_bins)
                agg_data = iv_woe_4iter(binned_data.copy(), target_col, class_col)
                remarks_list.append({"feature": c_i, "remarks": remarks})
            else:
                agg_data = iv_woe_4iter(data[[c_i, target_col]].copy(), target_col, c_i)
                remarks_list.append({"feature": c_i, "remarks": "categorical"})
            # print("---{} seconds. c_i: {}----".format(round(time.time() - c_i_start_time, 2), c_i))
            woe_iv = woe_iv.append(agg_data)
    return woe_iv, pd.DataFrame(remarks_list)

In [8]:
def get_iv_woe(data, target_col, max_bins):
    func_start_time = time.time()
    woe_iv, binning_remarks = var_iter(data, target_col, max_bins)
    print("------------------IV and WOE calculated for individual groups.------------------")
    print("Total time elapsed: {} minutes".format(round((time.time() - func_start_time) / 60, 3)))
    
    woe_iv["feature"] = woe_iv["feature"].replace("_bins", "", regex=True)    
    woe_iv = woe_iv[["feature", "sample_class", "sample_class_label", "sample_count", "min_value", "max_value",
                     "non_event_count", "non_event_rate", "event_count", "event_rate", 'distbn_non_event',
                     'distbn_event', 'woe', 'iv']]
    
    iv = woe_iv.groupby("feature")[["iv"]].agg(["sum", "count"]).reset_index()
    print("------------------Aggregated IV values for features calculated.------------------")
    print("Total time elapsed: {} minutes".format(round((time.time() - func_start_time) / 60, 3)))
    
    iv.columns = ["feature", "iv", "number_of_classes"]
    null_percent_data=pd.DataFrame(data.isnull().mean()).reset_index()
    null_percent_data.columns=["feature", "feature_null_percent"]
    iv=iv.merge(null_percent_data, on="feature", how="left")
    print("------------------Null percent calculated in features.------------------")
    print("Total time elapsed: {} minutes".format(round((time.time() - func_start_time) / 60, 3)))
    iv = iv.merge(binning_remarks, on="feature", how="left")
    woe_iv = woe_iv.merge(iv[["feature", "iv", "remarks"]].rename(columns={"iv": "iv_sum"}), on="feature", how="left")
    print("------------------Binning remarks added and process is complete.------------------")
    print("Total time elapsed: {} minutes".format(round((time.time() - func_start_time) / 60, 3)))
    return iv, woe_iv.replace({"Missing": np.nan})

In [9]:
iv, woe_iv = get_iv_woe(df.copy(), target_col="target", max_bins=20)

C:\Users\brian\AppData\Local\Temp\ipykernel_28100\2063493339.py:9: PerformanceWarning: indexing past lexsort depth may impact performance.
  temp_groupby = binned_data.groupby(class_col).agg({class_col:["first", "first"],
c:\Users\brian\AppData\Local\Programs\Python\Python38\lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Users\brian\AppData\Local\Temp\ipykernel_28100\815430266.py:21: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  woe_iv = woe_iv.append(agg_data)
C:\Users\brian\AppData\Local\Temp\ipykernel_28100\2063493339.py:9: PerformanceWarning: indexing past lexsort depth may impact performance.
  temp_groupby = binned_data.groupby(class_col).agg({class_col:["first", "first"],
c:\Users\brian\AppData\Local\Programs\Python\Python38\lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning

------------------IV and WOE calculated for individual groups.------------------
Total time elapsed: 1.134 minutes
------------------Aggregated IV values for features calculated.------------------
Total time elapsed: 1.135 minutes
------------------Null percent calculated in features.------------------
Total time elapsed: 1.135 minutes
------------------Binning remarks added and process is complete.------------------
Total time elapsed: 1.135 minutes


C:\Users\brian\AppData\Local\Temp\ipykernel_28100\815430266.py:21: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  woe_iv = woe_iv.append(agg_data)
C:\Users\brian\AppData\Local\Temp\ipykernel_28100\815430266.py:21: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  woe_iv = woe_iv.append(agg_data)


In [10]:
iv

,feature,iv,number_of_classes,feature_null_percent,remarks
0,avgNumDCLitigation,0.000000,1,0.0,categorical
1,avgNumHCLitigation,0.000000,1,0.0,categorical
2,avgNumMCLitigation,0.000000,1,0.0,categorical
3,avgNumOSOAOPOMLitigation,0.000000,1,0.0,categorical
4,avgNumWindingUpLitigation,0.000000,1,0.0,categorical
...,...,...,...,...,...
5896,totalamountOfIndividualLitigation_Others_SC / ...,0.000522,2,0.0,categorical
5897,totalamountOfIndividualLitigation_Others_SC / ...,0.000522,2,0.0,categorical
5898,totalamountOfIndividualLitigation_Others_SC / ...,0.000000,1,0.0,categorical
5899,totalamountOfIndividualLitigation_Others_SC / ...,0.000000,1,0.0,categorical


In [11]:
woe_iv

,feature,sample_class,sample_class_label,sample_count,min_value,max_value,non_event_count,non_event_rate,event_count,event_rate,distbn_non_event,distbn_event,woe,iv,iv_sum,remarks
0,entityNo,00122780R,NaN,1,00122780R,00122780R,0,0.000000,1,1.000000,0.000000,0.022727,0.000000,0.000000,0.000000,categorical
1,entityNo,07819469E,NaN,1,07819469E,07819469E,1,1.000000,0,0.000000,0.043478,0.000000,0.000000,0.000000,0.000000,categorical
2,entityNo,11082468E,NaN,1,11082468E,11082468E,1,1.000000,0,0.000000,0.043478,0.000000,0.000000,0.000000,0.000000,categorical
3,entityNo,14185777Z,NaN,1,14185777Z,14185777Z,0,0.000000,1,1.000000,0.000000,0.022727,0.000000,0.000000,0.000000,categorical
4,entityNo,17151733Y,NaN,1,17151733Y,17151733Y,1,1.000000,0,0.000000,0.043478,0.000000,0.000000,0.000000,0.000000,categorical
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7476,totalamountOfBankruptcyRecord_Others_Within36M...,"(63606.8, 162223.0]",1.0,14,65006.0,162223.0,6,0.428571,8,0.571429,0.260870,0.181818,0.361013,0.028539,0.036571,binned monotonically
7477,totalamountOfBankruptcyRecord_Others_Within36M...,NaN,NaN,0,NaN,NaN,0,NaN,0,NaN,0.000000,0.000000,NaN,NaN,0.036571,binned monotonically
7478,numberOfBankruptcyRecord_Others_Within36Months...,"(-0.001, 1.0]",0.0,59,0.0,1.0,20,0.338983,39,0.661017,0.869565,0.886364,-0.019134,0.000321,0.002637,binned monotonically
7479,numberOfBankruptcyRecord_Others_Within36Months...,"(1.0, 2.0]",1.0,8,2.0,2.0,3,0.375000,5,0.625000,0.130435,0.113636,0.137870,0.002316,0.002637,binned monotonically


In [12]:
selected_iv = iv[iv["iv"]>0.02]
selected_iv = selected_iv[selected_iv["remarks"]!="categorical"]
selected_iv

,feature,iv,number_of_classes,feature_null_percent,remarks
10,numActiveChargesAgainstNumOtherBusinessActiveC...,0.161592,2,0.0,binned forcefully
12,numAlternateDirectorsAgainstNumOtherBusinessAl...,0.090513,3,0.0,binned monotonically
13,numCurrentDissolvedBusinesses,0.098033,3,0.0,binned monotonically
18,numDirectorsAgainstNumOtherBusinessDirectors,0.051225,2,0.0,binned forcefully
20,numKeyPositions,0.309453,3,0.0,binned monotonically
...,...,...,...,...,...
5738,totalamountOfIndividualLitigation_Others_MC / ...,0.123235,3,0.0,binned monotonically
5748,totalamountOfIndividualLitigation_Others_MC / ...,0.123235,3,0.0,binned monotonically
5758,totalamountOfIndividualLitigation_Others_MC / ...,0.123235,3,0.0,binned monotonically
5768,totalamountOfIndividualLitigation_Others_MC / ...,0.123235,3,0.0,binned monotonically


In [13]:
!pip install varclushi

  Using cached varclushi-0.1.0-py3-none-any.whl (6.3 kB)
  Using cached factor_analyzer-0.3.1-py3-none-any.whl (41 kB)


You should consider upgrading via the 'c:\users\brian\appdata\local\programs\python\python38\python.exe -m pip install --upgrade pip' command.


In [14]:
from varclushi import VarClusHi

clus_df = df[selected_iv["feature"].values.tolist()]
var_clus_model = VarClusHi(clus_df, maxeigval2 = 0.7, maxclus=20)
var_clus_model.varclus()

In [15]:
np.corrcoef(df["target"], df["recentYearIndividualServicesSearchCount"])[0,1]

0.20218252664634853

In [16]:
var_clus_df = var_clus_model.rsquare
var_clus_df["iv"] = [selected_iv[selected_iv["feature"]==i]["iv"].values[0] for i in var_clus_df["Variable"]]
#var_clus_df["corr"] = [np.corrcoef(df["target"], df[c])[0,1] for c in var_clus_df["Variable"]]
var_clus_df.head()

,Cluster,Variable,RS_Own,RS_NC,RS_Ratio,iv
0,0,numCurrentDissolvedBusinesses,0.844536,0.485299,0.302047,0.098033
1,0,totalamountOfIndividualLitigation_Others_MC / ...,0.996340,0.708054,0.012538,0.123235
2,0,totalamountOfIndividualLitigation_Others_MC / ...,0.996340,0.708054,0.012538,0.123235
3,0,totalamountOfIndividualLitigation_Others_MC / ...,0.996340,0.708054,0.012538,0.123235
4,0,totalamountOfIndividualLitigation_Others_MC / ...,0.996340,0.708054,0.012538,0.123235


In [17]:
var_clus_df.sort_values(by=["Cluster", "iv"], ascending=[True, False], inplace=True)
var_clus_df.head()

,Cluster,Variable,RS_Own,RS_NC,RS_Ratio,iv
1,0,totalamountOfIndividualLitigation_Others_MC / ...,0.99634,0.708054,0.012538,0.123235
2,0,totalamountOfIndividualLitigation_Others_MC / ...,0.99634,0.708054,0.012538,0.123235
3,0,totalamountOfIndividualLitigation_Others_MC / ...,0.99634,0.708054,0.012538,0.123235
4,0,totalamountOfIndividualLitigation_Others_MC / ...,0.99634,0.708054,0.012538,0.123235
5,0,totalamountOfIndividualLitigation_Others_MC / ...,0.99634,0.708054,0.012538,0.123235


In [18]:
var_clus_df.groupby('Cluster').head(1)

,Cluster,Variable,RS_Own,RS_NC,RS_Ratio,iv
1,0,totalamountOfIndividualLitigation_Others_MC / ...,0.996340,0.708054,1.253819e-02,0.123235
7,1,numSubsidiary,0.149503,0.052626,8.977419e-01,0.099502
20,2,numKeyPositions,0.142680,0.092525,9.447313e-01,0.309453
28,3,totalamountOfBusinessLitigation_Discharged_MC ...,1.000000,0.557757,0.000000e+00,0.333071
34,4,numberOfBankruptcyRecord_Concluded_Within12Mon...,0.931397,0.197601,8.549691e-02,0.049511
46,5,recentYearIndividualServicesSearchCount,0.230459,0.639509,2.134701e+00,0.119889
53,6,totalamountOfBankruptcyRecord_Discharged_Withi...,1.000000,0.145161,0.000000e+00,0.049511
65,7,recentYearIndividualHirePurchaseSearchCount,0.537904,0.466334,8.658906e-01,0.169654
66,8,totalamountOfIndividualLitigation_Bankruptcy R...,1.000000,0.247109,5.898451e-16,0.021384
72,9,totalamountOfIndividualLitigation_Concluded_SC...,1.000000,0.107874,0.000000e+00,0.200775


In [19]:
selected_var_list = var_clus_df.groupby('Cluster').head(1)["Variable"].values.tolist()
selected_var_list

['totalamountOfIndividualLitigation_Others_MC / WOS_Within12Months_Individualplantiff_DueToNegligence & Accidents',
 'numSubsidiary',
 'numKeyPositions',
 'totalamountOfBusinessLitigation_Discharged_MC / OC_Within12Months_Companyplantiff_DueToContracts',
 'numberOfBankruptcyRecord_Concluded_Within12Months_Individualplantiff',
 'recentYearIndividualServicesSearchCount',
 'totalamountOfBankruptcyRecord_Discharged_Within12Months_Companyplantiff',
 'recentYearIndividualHirePurchaseSearchCount',
 'totalamountOfIndividualLitigation_Bankruptcy Related_SC / OC_Within12Months_Individualplantiff_DueToOthers',
 'totalamountOfIndividualLitigation_Concluded_SC / WOS_Within12Months_Individualplantiff_DueToNegligence & Accidents',
 'totalamountOfBankruptcyRecord_Concluded_Within12Months_Companyplantiff',
 'totalamountOfBankruptcyRecord_Discharged_Within12Months_Individualplantiff',
 'totalamountOfBankruptcyRecord_Bankruptcy Related_Within12Months_Individualplantiff',
 'totalamountOfIndividualLitigati

In [20]:
selected_var_df = var_clus_df.groupby('Cluster').head(1)
selected_var_df.to_csv('../selected_features/ibs_selected_features.csv', index=False)